In [ ]:
import xml.etree.ElementTree as et
import pandas as pd
import re
import os
from iso3166 import countries


In [ ]:
%load_ext autoreload
%autoreload 2
import src.xml_parse as parse_utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
DATA_PATH = "/gpfs01/berens/data/data/pubmed_central/baseline_2026-01-23"
SECTION = "PMC006xxxxxx_comm"
FILENAME = "PMC8211483.xml"

In [ ]:
xml_file = os.path.join(DATA_PATH, SECTION, FILENAME)
xtree = et.parse(xml_file, parser=et.XMLParser(encoding="UTF-8"))
xroot = xtree.getroot()

In [ ]:
for child in xroot:
    print(child.tag, child.attrib)
    for child2 in child:
        print("    ", child2.tag, child2.attrib)
        for child3 in child2:
            print("        ", child3.tag, child3.attrib)

In [ ]:
xroot.tag

In [ ]:
body = xroot.findall("./body")
if len(body) > 0:
    body = body[0]
body

In [ ]:
secs = xroot.findall("./body/sec")
len(secs)

In [ ]:
for sec in secs:
    text = " ".join(sec.itertext())
text

'\nDespite multiple limitations of this study, to include its retrospective review of a database that provides a suboptimal diagnosis of PEP and fails to account for most procedural and endoscopist risk factors, it nevertheless is very clear that endoscopists in this database failed to adhere to societal guidelines to reduce the risk of PEP in two-thirds of the patients. These include recommendations by the ASGE and ESGE as well as from Japan\n 3 \n 4 \n 5 \n.\n \nGiven the prospective and randomized controlled trials demonstrating the efficacy of rectal NSAIDs to reduce the risk and severity of PEP in average as well as high-risk patients\n 6 \n 7 \n 8 \n 9 \n 10 \n 11 \n, these societal guidelines are clear that all patients undergoing ERCP should receive periprocedural rectal NSAIDS.\u200aMoreover, although there are no formal societal guidelines that recommend use of PPS, their use has been associated with a decreased risk of PEP in patients undergoing complex ERCP\n 10 \n. Such stents were placed in 4.3\u200a% of these patients in this series, and only 1.6\u200a% of these patients had both PPS and rectal indomethacin. Whether this “belt and suspenders” prophylaxis proves more effective than either technique alone to prevent PEP is unknown at the time but is currently undergoing study in the National Institutes of Health-sponsored stent vs indomethacin study (SVI).\n \nSo, what good are societal guidelines, most derived from a graded review of the literature, supplemented by expert opinion\n 12 \n 13 \n 14 \n 15 \n, if they are not followed? And is this underutilization occasional or systemic? Furthermore, are there any repercussions of failure to follow them other than the potential for suboptimal patient outcomes? In other words, do they have any teeth such that failure to follow them leads to failure to become privileged in an endoscopy unit or censure of privilege by ignoring one or more specific guidelines? Is there a financial incentive to adhere to guidelines, or is there actually a perverse incentive to overuse certain endoscopic procedures despite guidelines?\n The above are complex issues and perhaps beyond the scope of this editorial. \nGuideline adherents point to improved patient outcomes and standardization of care, whereas those who fail to heed them cite contradictory recommendations, the continued evolution of guideline recommendations, and their profusion in all aspects of endoscopic practice leading to “cookbook care.” As such, we have evolved practice patterns to ensure compliance with screening and potentially banding cirrhotic patients to prevent an index variceal bleed\n 16 \n. We recommend screening of virtually all patients for colorectal cancer and variable surveillance intervals based upon initial findings, genetics, and family history\n 17 \n 18 \n 19 \n 20 \n 21 \n. We screen and variably surveille for Barrett’s esophagus and undertake or refer patients to experienced colleagues for endoscopic mucosal resection or endoscopic submucosal dissection with finding of high-grade dysplasia and high-risk patients with superficial malignancy\n 22 \n 23 \n 24 \n 25 \n 26 \n. We recognize that chromo-endoscopy finds more dysplasia with fewer biopsies then white light endoscopy with or without the ability to magnify the image or filter the wavelength, but without formal societal guidelines, it is the minority of endoscopists who utilize it in screening patients with inflammatory bowel disease\n 27 \n 28 \n 29 \n. However, there are guidelines to screen for gastric malignancy\n 30 \n, to perform and interpret capsule endoscopy\n 31 \nappropriately, and about the appropriate application of endoscopic ultrasound\n 32 \n, as well as dozens of other endoscopic procedures. Not to mention procedural sedation, the appropriate cessation intervals for various anticoagulants and the indications for periprocedural antibiotics.\n \nThat brings us back to the Issak et al manuscript in the current issue of Endoscopy International Open\n 1 \n. Was failure to treat patients undergoing ERCP with NSAIDs or PSP related to skepticism about the results of previous studies, ignorance of those studies, or as the authors suggest, inadequate access to rectal indomethacin (or skill set to place a PSP)? It is clear, at least in this instance, that underutilization of societal guidelines is widespread and that this failure can be associated with significant iatrogenic patient illness on the one hand and cumulative hospitalization-related expense, on the other. Without definitive consequences for non-adherence, guidelines are just that, guidelines.\n'

In [ ]:
sections_dict = {}
for sec in secs:
    text = " ".join(sec.itertext())
    for child in sec:
        # only parse sections with title, to avoid mismatch
        if child.tag == "title":
            title = child.text
            sections_dict[title] = text

sections_dict

In [ ]:
# if there are more sections than section titles, and the first section
# is untitled, it is assumed to be the introduction
first_sec = " ".join(secs[0].itertext())
first_sec_in_dict = list(sections_dict.values())[0]
if len(sections_dict) < len(secs):
    if not first_sec == first_sec_in_dict:
        sections_dict["Introduction"] = first_sec

sections_dict

In [ ]:
# get the first text of the body (which is not inside of a section)
# if this exists, it is assumed to be the introduction
# this can cause problems, e.g. in "PMC8483461.xml", where this part is
# not the introduction, but the abbreviations and highlights
# in other papers, this includes the introduction, but also other
# such as key insights
intro = ""
for child in body:
    if child.tag == "sec":
        break
    intro += " ".join(child.itertext())
if len(intro) > 0:
    sections_dict["Introduction"] = intro

sections_dict

{'Introduction': '\n \nThe current publication by Issak et al, “Prophylactic rectal indomethacin and pancreatic duct stents (PPS) for prevention of post-ERCP pancreatitis (PEP) are underutilized in average and high-risk patients undergoing ERCP”\n 1 \n, reviews 31,050 adults captured by the IBM Explorys database between 2014 and 2019.\u200aPatients were categorized by risk factors for PEP, which included female sex, age <\u200a40 years, sphincter of Oddi dysfunction, history of acute pancreatitis, or pancreatic sphincterotomy at time of the procedure\n 2 \n. Average-risk patients had no risk factors, and patients were stratified for 0 to ≥\u200a3 risk factors. The database did not allow retrieval of other procedural risk factors for PEP to include multiple cannulation attempts, multiple pancreatic duct injections, pancreatic acinarization with contrast injection, or failure to use guidewire cannulation to selectively access the biliary tree; nor did it define the ERCP experience of the endoscopist or whether a trainee was involved in the procedure. The primary outcome of the study was to define the incidence of nonsteroidal anti-inflammatory drug (NSAID) or pancreatic stent use in an attempt to decrease PEP. Secondary outcomes were prophylaxis for and incidence of PEP contingent upon number of risk factors. Not surprising was that as the number of risk factors increased, so too did PEP. What perhaps was surprising in this study was that only one-third of all patients undergoing ERCP received PEP prophylaxis. This included 82.4\u200a% receiving rectal indomethacin and 12.4\u200a% who had placement of a prophylactic pancreatic stent (PPS).\n'}


In [ ]:
list(sections_dict.keys())

In [ ]:
list(sections_dict.values())

In [ ]:
parse_utils.xml_get_text([secs[0]])

In [ ]:
titles = xroot.findall("./body/sec/title")
len(titles)

In [ ]:
title_names = parse_utils.get_sec_titles_old(xroot)
title_names

In [ ]:
parse_utils.xml_get_attr(xroot, "{http://www.w3.org/XML/1998/namespace}lang")

In [ ]:
parse_utils.xml_get_attr(xroot, "article-type")

In [ ]:
parse_utils.xml_get_text(
    xroot.findall("./front/journal-meta/journal-title-group/journal-title")
)

In [ ]:
# not all articles have pmid, so also collect pmc id
parse_utils.xml_get_text(
    xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmid"]')
)

In [ ]:
parse_utils.xml_get_text(
    xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmc"]')
)

In [ ]:
parse_utils.xml_get_text(
    xroot.findall("./front/article-meta/title-group/article-title")
)

In [ ]:
parse_utils.get_country(xroot)

In [ ]:
parse_utils.get_date(xroot)

In [ ]:
# alternative version: get date dict to include tags to make sure day and month
# aren't confused
nodes = xroot.findall('./front/article-meta/pub-date/[@pub-type="epub"]')
date = {}
if len(nodes) > 0:
    for child in nodes[0]:
        date[child.tag] = child.text
date

In [ ]:
parse_utils.get_abstr(xroot.findall("./front/article-meta/abstract"))

In [ ]:
parse_utils.xml_parse_single(xml_file)

In [ ]:
df_2026 = pd.read_json(os.path.join("../data/baseline_2026-01-23", SECTION + ".json"))
df_2026

In [ ]:
df_2025 = pd.read_json(os.path.join("../data/baseline_2025-06-26", SECTION + ".json"))
df_2025

In [ ]:
ids = list(df_2026["pmc-id"])
for id in df_2025["pmc-id"]:
    if not id in ids:
        print(id)
        break